# MilliRota — GPU Eğitim (Colab)
8 epoch'tan kaldığı yerden devam eder. Runtime → Change runtime type → **T4 GPU** seç.

In [ ]:
# 1. Bağımlılıklar
!pip install ultralytics -q

In [ ]:
# 2. millirota_8ep_last.pt dosyasını ve veri setini yükle
# Sol menüden (klasör ikonu) dosyaları sürükle-bırak yapabilirsin
# VEYA Google Drive'dan:
from google.colab import drive
drive.mount('/content/drive')
# !cp /content/drive/MyDrive/millirota_8ep_last.pt /content/
# !cp -r /content/drive/MyDrive/millirota_dataset /content/

In [ ]:
# 3. millirota.yaml oluştur (veri seti yolu Colab için güncelle)
yaml_content = '''
path: /content/millirota_dataset
train: train/images
val:   val/images
nc: 7
names:
  0: insan
  1: tasit_hareketli
  2: tasit_hareketsiz
  3: uai_uygun
  4: uai_uygun_degil
  5: uap_uygun
  6: uap_uygun_degil
'''
with open('/content/millirota_colab.yaml', 'w') as f:
    f.write(yaml_content)
print('yaml hazır')

In [ ]:
# 4. GPU'da eğitimi başlat — 8 epoch'tan devam eder
from ultralytics import YOLO
model = YOLO('/content/millirota_8ep_last.pt')  # 8 epoch checkpoint
results = model.train(
    data='/content/millirota_colab.yaml',
    epochs=150,           # toplam 150 epoch hedefi
    imgsz=640,            # GPU'da yüksek çözünürlük
    batch=16,
    device='cuda',
    project='/content/millirota_runs',
    name='gpu_train',
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=0,      # 8 epoch zaten geçti
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    flipud=0.1, fliplr=0.5, mosaic=1.0, mixup=0.1,
    save=True, save_period=10,
    plots=True,
    patience=30,
)
print('Eğitim tamamlandı!')
print(f'En iyi model: /content/millirota_runs/gpu_train/weights/best.pt')

In [ ]:
# 5. En iyi modeli Drive'a kaydet
import shutil
shutil.copy('/content/millirota_runs/gpu_train/weights/best.pt',
            '/content/drive/MyDrive/millirota_best.pt')
print('Drive\'a kaydedildi: millirota_best.pt')